Verify Parquet Structure

In [1]:
import pandas as pd

In [2]:
caiso = pd.read_parquet("Datasets/CAISO/caiso_with_residuals.parquet")

In [3]:
caiso

,HR,PGE,SCE,SDGE,VEA,CAISO,seconds,day_index_3am,time,residuals,fitted,y
0,0,9844.620265,9928.001620,2088.453772,92.069996,21953.146163,1546300800,0,2018-12-31 17:00:00-07:00,0.000000,0.000000,0.000000
1,1,9507.691745,9604.449654,1998.915397,92.659879,21203.716174,1546304400,0,2018-12-31 18:00:00-07:00,0.000000,0.000000,0.000000
2,2,9251.837855,9340.780790,1934.057172,93.592896,20620.268525,1546308000,0,2018-12-31 19:00:00-07:00,0.000000,0.000000,0.000000
3,3,9123.916881,9183.897175,1891.808751,97.052299,20296.675172,1546311600,0,2018-12-31 20:00:00-07:00,0.000000,0.000000,0.000000
4,4,9156.274437,9170.727964,1898.794915,101.469059,20327.266153,1546315200,1,2018-12-31 21:00:00-07:00,0.511937,0.000083,0.512021
...,...,...,...,...,...,...,...,...,...,...,...,...
56946,19,16282.880880,16406.503620,2982.160214,121.460361,35793.005193,1751331600,2373,2025-06-30 13:00:00-06:00,-0.014508,1.633906,1.619398
56947,20,15913.359630,15813.555210,2934.673798,110.556472,34772.145278,1751335200,2373,2025-06-30 14:00:00-06:00,0.002201,1.489926,1.492127
56948,21,15324.313130,15213.739570,2853.128183,98.707809,33489.888495,1751338800,2373,2025-06-30 15:00:00-06:00,0.057379,1.274887,1.332267
56949,22,14194.494270,14301.485070,2710.082027,86.417304,31292.478469,1751342400,2373,2025-06-30 16:00:00-06:00,0.018423,1.039891,1.058314


In [4]:
caiso_MW = caiso['CAISO']/1000

In [5]:
caiso_beast = pd.concat([caiso['time'], caiso['CAISO']], axis=1)

In [6]:
caiso_beast

,time,CAISO
0,2018-12-31 17:00:00-07:00,21953.146163
1,2018-12-31 18:00:00-07:00,21203.716174
2,2018-12-31 19:00:00-07:00,20620.268525
3,2018-12-31 20:00:00-07:00,20296.675172
4,2018-12-31 21:00:00-07:00,20327.266153
...,...,...
56946,2025-06-30 13:00:00-06:00,35793.005193
56947,2025-06-30 14:00:00-06:00,34772.145278
56948,2025-06-30 15:00:00-06:00,33489.888495
56949,2025-06-30 16:00:00-06:00,31292.478469


In [7]:
caiso_beast.to_csv("CAISO_MW.csv")

In [8]:
caiso_beast = caiso_beast.rename(columns={'time':'start_time', 'CAISO' : 'tensor'})

In [9]:
import pickle
import zipfile
import pgzip
import os

In [10]:
path = os.getcwd()

In [11]:
path

'/home/lambpc/AMARANTH/opensourcegridmodeling/Transformer/code/code'

In [12]:
with pgzip.open(os.path.join(path,'Datasets/PJM_energy_datasets/dayton_dict.pkl.pgz')) as f:
            sd = pickle.load(f)

In [13]:
sd

{'start_time': datetime.datetime(2004, 10, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400))),
 'tensor': tensor([1621., 1536., 1500.,  ..., 2405., 2250., 2042.])}

In [14]:
len(sd['tensor'])

121296

In [15]:
import torch
torch.stack([sd['tensor']]).unsqueeze(-1)

tensor([[[1621.],
         [1536.],
         [1500.],
         ...,
         [2405.],
         [2250.],
         [2042.]]])

In [16]:
pd.DataFrame(sd)

,start_time,tensor
0,2004-10-01 00:00:00-05:00,1621.0
1,2004-10-01 00:00:00-05:00,1536.0
2,2004-10-01 00:00:00-05:00,1500.0
3,2004-10-01 00:00:00-05:00,1434.0
4,2004-10-01 00:00:00-05:00,1489.0
...,...,...
121291,2004-10-01 00:00:00-05:00,2554.0
121292,2004-10-01 00:00:00-05:00,2481.0
121293,2004-10-01 00:00:00-05:00,2405.0
121294,2004-10-01 00:00:00-05:00,2250.0


In [17]:
time_needed = sd['start_time']

In [18]:
import datetime
tensor = torch.from_numpy(caiso_beast['tensor'].values)

In [19]:
dict = {'start_time' : datetime.datetime(2018,12,31,0,0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200))), 'tensor' : tensor}

In [20]:
len(dict['tensor'])

56951

In [21]:
with open('/home/lambpc/AMARANTH/opensourcegridmodeling/Transformer/code/code/Datasets/CAISO/caiso_dict.pkl','wb') as f :
    pickle.dump(dict,f)

In [22]:
with open(os.path.join(path,'Datasets/CAISO/caiso_dict.pkl'),'rb') as f :
    sd = pickle.load(f)

In [23]:
with pgzip.open(os.path.join(path,'Datasets/CAISO/caiso_dict.pkl.pgz'), 'wb') as f:
            pickle.dump(sd,f)

In [24]:
with pgzip.open(os.path.join(path,'Datasets/CAISO/caiso_dict.pkl.pgz')) as f:
            sd = pickle.load(f)

In [25]:
sd

{'start_time': datetime.datetime(2018, 12, 31, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200))),
 'tensor': tensor([21953.1462, 21203.7162, 20620.2685,  ..., 33489.8885, 31292.4785,
         28781.0817], dtype=torch.float64)}

In [26]:
len(sd['tensor'])

56951

In [27]:
type(sd['tensor'])

torch.Tensor

In [28]:
torch.stack([sd['tensor']]).unsqueeze(-1)

tensor([[[21953.1462],
         [21203.7162],
         [20620.2685],
         ...,
         [33489.8885],
         [31292.4785],
         [28781.0817]]], dtype=torch.float64)

In [29]:
start_idx = 0
end_idx = 9999999
seq_len = 816
pred_horz = 24
stride=-1
timestamp = True

with pgzip.open(os.path.join(path,'Datasets/CAISO/caiso_dict.pkl.pgz')) as f:
            sd = pickle.load(f)

In [30]:
series = sd["tensor"] #121296 total elements
starttime = sd['start_time']
        
total_datapoints = len(series)
        
end_idx = min(end_idx,len(series))

In [31]:
series

tensor([21953.1462, 21203.7162, 20620.2685,  ..., 33489.8885, 31292.4785,
        28781.0817], dtype=torch.float64)

In [32]:
starttime + -13046 *datetime.timedelta(hours=1)

datetime.datetime(2017, 7, 5, 10, 0, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200)))

In [33]:
attr_dset_smpl_rt = 24 #Samples per day. Spain, AEP: 24, London: 48 
param_dset_lookback_weeks = 5
param_dset_forecast = 48

param_dset_lookback = param_dset_lookback_weeks*7*attr_dset_smpl_rt - param_dset_forecast

seq_len = param_dset_lookback
pred_horz = param_dset_forecast

In [34]:
seq_len

792

In [35]:
pred_horz

48

In [36]:
path = os.getcwd()

In [37]:
from Datasets.CAISO.caiso_def import CAISO
full_set = CAISO(path = os.path.join(path + "/Datasets/CAISO"),
                      seq_len = param_dset_lookback,
                      pred_horz = param_dset_forecast,
                      timestamp = False)
dytmax = full_set.max()
dytmin = full_set.min()
del(full_set)
    
train_set = CAISO(path = os.path.join(path + "/Datasets/CAISO"),
                    start_idx = 0, end_idx = 18983,
                    seq_len = param_dset_lookback,
                    pred_horz = param_dset_forecast,
                    timestamp = False)
val_set = CAISO(path = os.path.join(path + "/Datasets/CAISO"),
                  start_idx = 18983, end_idx = 2*18983,
                    seq_len = param_dset_lookback,
                    pred_horz = param_dset_forecast,
                    timestamp = False)
test_set = CAISO(path = os.path.join(path + "/Datasets/CAISO"),
                    start_idx = 2*18983,
                    seq_len = param_dset_lookback,
                    pred_horz = param_dset_forecast,
                    timestamp = False)
    
train_set.series = (train_set.series.half() - dytmin)/(dytmax - dytmin)
val_set.series = (val_set.series.half() - dytmin)/(dytmax - dytmin)
test_set.series = (test_set.series.half() - dytmin)/(dytmax - dytmin)

In [38]:
#Transformer only parameters
param_trf_edim = 24
param_trf_heads = 4
param_trf_elyr = 4
param_trf_dlyr = 4
param_trf_ffdim = 256
param_trf_weather = False

#Bigbird only parameters
param_trf_bksz = 48

#Training Params
arg_ini_lr = 1e-4
arg_batchsz = 'auto'
arg_epochs = 1000

custom_collate = torch.utils.data.default_collate

In [39]:
train_loader = torch.utils.data.DataLoader(train_set,batch_size=32,
                                           shuffle=True, collate_fn= custom_collate,
                                           pin_memory=True)

In [40]:
train_loader

In [41]:
train_loader.dataset.series

tensor([[[0.1956],
         [0.1750],
         [0.1593],
         ...,
         [0.3137],
         [0.2634],
         [0.2203]],

        [[0.1929],
         [0.1746],
         [0.1642],
         ...,
         [0.3088],
         [0.2593],
         [0.2191]],

        [[0.1934],
         [0.1694],
         [0.1523],
         ...,
         [0.2961],
         [0.2430],
         [0.1942]],

        ...,

        [[0.1262],
         [0.1162],
         [0.1122],
         ...,
         [0.3132],
         [0.2644],
         [0.2213]],

        [[0.1959],
         [0.1772],
         [0.1689],
         ...,
         [0.2783],
         [0.2378],
         [0.2000]],

        [[0.1755],
         [0.1572],
         [0.1459],
         ...,
         [0.2632],
         [0.2278],
         [0.1912]]], dtype=torch.float16)

In [42]:
scaler = torch.cuda.amp.GradScaler()

/var/tmp/pbs.4905179.sawtoothpbs/ipykernel_72959/2340218076.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [43]:
import sys
sys.path.append("./Models")
import os
os.system('')

0

In [44]:
from Models.tsrnn import TSRNN, DPTrainableTSRNN
from Models.BigBirdSparse.SparseTransformerBB import TransformerBBSparse, TransformerBBFixed
from Models.transformer_base import Transformer_Base, DPTrainable

In [45]:
cuda_lead = torch.device("cuda:0")

In [46]:

model = TSRNN(smpl_rate = attr_dset_smpl_rt,
                  pred_horz = param_dset_forecast,
                  num_weeks = param_dset_lookback_weeks)
dpm = DPTrainableTSRNN(model,cuda_devices=None)
model.to(cuda_lead)

TSRNN(
  (full_rnn_layers): ModuleList(
    (0): Eidetic_LSTM(
      (inp_conv): Conv3d(1, 448, kernel_size=(3, 3, 2), stride=(1, 1, 1), padding=same)
      (hid_conv): Conv3d(64, 256, kernel_size=(3, 3, 2), stride=(1, 1, 1), padding=same)
      (gmem_conv): Conv3d(64, 256, kernel_size=(3, 3, 2), stride=(1, 1, 1), padding=same)
      (out_cell_conv): Conv3d(64, 64, kernel_size=(3, 3, 2), stride=(1, 1, 1), padding=same)
      (out_gmem_conv): Conv3d(64, 64, kernel_size=(3, 3, 2), stride=(1, 1, 1), padding=same)
      (out_memory_conv): Conv3d(128, 64, kernel_size=(1, 1, 1), stride=(1, 1, 1), padding=same)
      (hid_norm): LayerNorm((256, 2, 13, 24), eps=1e-05, elementwise_affine=True)
      (inp_norm): LayerNorm((448, 2, 13, 24), eps=1e-05, elementwise_affine=True)
      (cell_norm): LayerNorm((64, 2, 13, 24), eps=1e-05, elementwise_affine=True)
      (gmem_norm): LayerNorm((256, 2, 13, 24), eps=1e-05, elementwise_affine=True)
    )
    (1): Eidetic_LSTM(
      (inp_conv): Conv3d(64, 4

In [47]:
opt = torch.optim.AdamW(model.parameters(),lr = arg_ini_lr)

In [48]:
l,ls = dpm.train_epoch(train_loader, opt, device = cuda_lead, scaler = scaler)

/home/lambpc/AMARANTH/opensourcegridmodeling/Transformer/code/code/Models/tsrnn.py:312: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype = torch.bfloat16):
/home/lambpc/.conda/envs/AustinTXForecast/lib/python3.13/site-packages/torch/nn/modules/conv.py:712: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1027.)
  return F.conv3d(
/home/lambpc/.conda/envs/AustinTXForecast/lib/python3.13/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/lambpc/.conda/envs/AustinTXForecast/lib/python3.13/site-packages/numpy/_core/_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slic

In [49]:
dir(train_loader)

['_DataLoader__initialized',
 '_DataLoader__multiprocessing_context',
 '_IterableDataset_len_called',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_auto_collation',
 '_dataset_kind',
 '_get_iterator',
 '_index_sampler',
 '_iterator',
 'batch_sampler',
 'batch_size',
 'check_worker_number_rationality',
 'collate_fn',
 'dataset',
 'drop_last',
 'generator',
 'in_order',
 'multiprocessing_context',
 'num_workers',
 'persistent_workers',
 'pin_memory',
 'pin_memory_device',
 'prefetch_factor',
 'sampler'

In [55]:
val_loader = torch.utils.data.DataLoader(val_set,batch_size=32, collate_fn = custom_collate)
test_loader = torch.utils.data.DataLoader(test_set,batch_size=32, collate_fn = custom_collate)

In [57]:
import time
import copy
scaler = torch.cuda.amp.GradScaler()


best_model_state = None
best_val = 999999

epoch_start = 0
epoch_end = 0
avg_epoch_dur = 0
for i in range(arg_epochs):
    epoch_start = time.time_ns()
    
    l,ls = dpm.train_epoch(train_loader, opt, device = cuda_lead, scaler = scaler)    
    vl,vls = dpm.val(val_loader,
                     loss_fn = lambda x,y: torch.nn.MSELoss(reduction='none')(x, y).\
                             nanmean(dim=-2).\
                             sqrt_(),
                    device = cuda_lead)
    
    epoch_end = time.time_ns()
    vl_improved = (vl < best_val)
    if vl_improved:
        best_val = vl
        best_model_state = copy.deepcopy(model.state_dict())

    print("                                                  ",end="\r")
    if vl_improved:
        print("Epoch {0}: loss = {1}, val_loss =\033[1;32m {2} \033[1;37m".format(i,l,vl))
    else:
        print("Epoch {0}: loss = {1}, val_loss = {2}".format(i,l,vl))
    
    #Estimate completion time
    epoch_dur_sec = (epoch_end-epoch_start)/(1000000000)
    if i == 0:
        avg_epoch_dur = epoch_dur_sec
    avg_epoch_dur = 0.3*epoch_dur_sec + 0.7*avg_epoch_dur
    etc_sec = (arg_epochs - i - 1)*avg_epoch_dur
    est_hrs = int(etc_sec//3600)
    est_mins = round((etc_sec%3600)/60)
    if est_mins == 60:
        est_mins = 0
        est_hrs += 1
    print("Estimated time to complete: {0}h {1}m".format(est_hrs,est_mins),end='\r')

final_model_state = model.state_dict()
for key in final_model_state:
    final_model_state[key] = final_model_state[key].cpu().detach().numpy()

best_model_dpt = copy.deepcopy(dpm)
best_model_dpt.module.load_state_dict(best_model_state)


torch.cuda.empty_cache()
# def vis(model,dataset,device,idx=0):
#     model.eval()
#     (vis_src_, vis_tar_)  = dataset.__getitem__(idx)

#     src = vis_src_.detach().clone()
#     src = src.nan_to_num(0)
#     src = src.unsqueeze(0)

#     src = src.to(device)
#     out = model(src)
#     out_ = (dataset.max() - dataset.min())*out + dataset.min()
    
#     out_c = out_.cpu().detach()   

#     data = torch.cat((vis_src_,vis_tar_),dim=0)
#     data = (dataset.max() - dataset.min())*data + dataset.min()
#     disp_len1 = 336
#     plt.figure(dpi=400)
#     plt.plot(data[-disp_len1:],linewidth=0.5)
#     plt.plot(np.arange(disp_len1-48,disp_len1,1),out_c[0],linewidth=0.5)

#vis(best_model,test_set,torch.device("cuda:0"),idx=39)
    # disp_len2 = 336
    # plt.figure(dpi=240)
    # plt.plot(data[-disp_len2:],linewidth=0.5)
    # plt.plot(np.arange(disp_len2-24,disp_len2,1),out_c[0],linewidth=0.5)
losses = [ lambda x,y: torch.nn.MSELoss(reduction='none')(x, y).\
                          nanmean(dim=-2),
           lambda x,y: torch.nn.MSELoss(reduction='none')(x, y).\
                                     nanmean(dim=-2).sqrt_(),
           lambda x,t: (x-t).abs_().nanmean(dim=-2),
           lambda x,t: (2*(t-x).abs_() / (t.abs() + x.abs())).nanmean(dim=-2)]

test_loss , tls = dpm.val(test_loader,
                          loss_fn = losses,
                          device = cuda_lead)
print("Test loss (Final Epoch): {0}".format(test_loss))
test_loss , tls = best_model_dpt.val(test_loader,
                          loss_fn = losses,
                          device = cuda_lead)
print("Test loss (Best Validation): {0}".format(test_loss))

/var/tmp/pbs.4905179.sawtoothpbs/ipykernel_72959/3131824892.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 0: loss = 0.08469369262456894, val_loss = 0.15933562815189362 
Epoch 1: loss = 0.1107635647058487, val_loss = 0.08553681522607803 
Epoch 2: loss = 0.070893794298172, val_loss = 0.08045203238725662 
Epoch 3: loss = 0.047464899718761444, val_loss = 0.12078707665205002
Epoch 4: loss = 0.053828924894332886, val_loss = 0.12258543819189072
Epoch 5: loss = 0.05930565297603607, val_loss = 0.09107670187950134
Epoch 6: loss = 0.04858207702636719, val_loss = 0.06720919907093048 
Epoch 7: loss = 0.040037017315626144, val_loss = 0.07361084222793579
Epoch 8: loss = 0.03861689195036888, val_loss = 0.08030939847230911
Epoch 9: loss = 0.03831678256392479, val_loss = 0.06977880001068115
Epoch 10: loss = 0.03351122885942459, val_loss = 0.06015157327055931 
Epoch 11: loss = 0.02922239899635315, val_loss = 0.07003159075975418
Epoch 12: loss = 0.027103617787361145, val_loss = 0.0867639109492302
Epoch 13: loss = 0.030239850282669067, val_loss = 0.09347529709339142
Epoch 14: loss = 0.03191831335425377, 

In [64]:
def predict(loader: torch.utils.data.DataLoader, device):
    src_val = []
    tar_val = []
    out_val = []
    best_model_dpt.dpmod.eval()
            
    with torch.no_grad():
        for batch_idx, (src, tar) in enumerate(loader):
            src_ = src.to(device)
            tar_ = tar.to(device)
                
            src_ = src_.nan_to_num()
            out,_ = best_model_dpt.dpmod(src_)
                
            out_ = out #(25.695 - 9.581)*out + 9.581
                # loss = torch.nn.MSELoss(reduction='none')(out_, tar_).\
                #         nanmean(dim=-2).\
                #         sqrt_()

            src_val.append(src)
            tar_val.append(tar)
            out_val.append(out)
        
    return src_val, tar_val, out_val

In [181]:
import numpy as np
def predict_2(loader: torch.utils.data.DataLoader, device):
    src_val = []
    tar_val = []
    out_val = []
    
    best_model_dpt.dpmod.eval()

    with torch.no_grad():
        for batch_idx, (src,tar) in enumerate(loader):
            src_ = src.to(device)
            tar_ = tar.to(device)

            src_ = src_.nan_to_num()
            out,_ = best_model_dpt.dpmod(src_)
                
            out_ = out #(25.695 - 9.581)*out + 9.581
                # loss = torch.nn.MSELoss(reduction='none')(out_, tar_).\
                #         nanmean(dim=-2).\
                #         sqrt_()

                 # src_ is the predictions for input of src
                # tar_ is predictions for input of tar

                # To get the training preds for src, tar_val = src_

            for target in tar_:
                target = target.cpu()
                tar_val.append(target)

            for source in out_:
                source = source.cpu()
                src_val.append(source)

            for output in out_:
                output = output.cpu()
                out_val.append(output)

    
        tar_val = np.array(tar_val).flatten()
        src_val = np.array(src_val).flatten()
        out_val = np.array(out_val).flatten()
        
        return src_val, tar_val

In [182]:
src,tar = predict_2(train_loader, cuda_lead)

In [183]:
src.shape

(1056,)

In [184]:
tar.shape

(1056,)